# ORA-Q final run on Kaggle (T4)
Run top to bottom. Keep the GPU off until step 5 to save quota (Session options > Accelerator: GPU T4).

This is a new run. It writes `results_final.csv`, `results_final_meta.json` and `role_report_final.json`, and keeps the earlier results files as they are.

## 1. Install
Pins transformers to the version used for the CPU checks. Restart the session after this cell if Kaggle asks.

In [ ]:
!pip install -q "transformers==5.24.1" h5py
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 2. Get the code
Commit and push the `ora-q` branch first, then set `COMMIT` to that commit so the run is tied to it.

In [ ]:
COMMIT = "<full commit hash>"   # edit
!git clone -q -b ora-q https://github.com/maiHydrogen/DA4022.git /kaggle/working/DA4022
%cd /kaggle/working/DA4022
!git checkout -q $COMMIT && git log -1 --format="%H %s" && git status --short

## 3. Get the data
Use the official NYU links (internet ON), or a Kaggle dataset with the same files. The hashes must match the ones in `Dataloaders/evaluation/nyuv2.md`.

In [ ]:
!find /kaggle/input -name "*.mat" 2>/dev/null
!wget -q -O /kaggle/working/splits.mat http://horatio.cs.nyu.edu/mit/silberman/indoor_seg_sup/splits.mat
!wget -q -O /kaggle/working/nyu_depth_v2_labeled.mat http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat

In [ ]:
MAT = "/kaggle/working/nyu_depth_v2_labeled.mat"   # edit if you use a Kaggle dataset
SPLITS = "/kaggle/working/splits.mat"
EXPECTED = {MAT: "2d724b0c0ab358aa1ce5df855e5bd14a2279ab7202efe22f32a30d612dcb86aa",
            SPLITS: "6e081404491a8bfba2f066beaa9713ef6046f9007aa5f799f2a350506a580cee"}
import hashlib
for path, want in EXPECTED.items():
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    print(path, "OK" if h.hexdigest() == want else "MISMATCH " + h.hexdigest())

## 4. Unit tests (CPU is fine)

In [ ]:
!python -m pytest -q -p no:cacheprovider Quantization_and_Pruning/tests

## 5. Quick checks on 20 images (turn the GPU on now)
FP32 gave 0.9584 on these 20 images in our CPU check. The 16-bit run must match FP32 closely; if not, stop.

In [ ]:
R = "Quantization_and_Pruning/experiments/ora_q/run_ora_q.py"
E = "Quantization_and_Pruning/experiments/ora_q"
REV = "5426e4f0f36572d16453bbda7a8389317b1bef99"
!python $R --mat $MAT --splits $SPLITS --revision $REV --methods fp32 --num-images 20 --no-latency --out /kaggle/working/quick.csv
!python $R --mat $MAT --splits $SPLITS --revision $REV --methods minmax --num-images 20 --act-bits 16 --weight-bits 16 --no-latency --out /kaggle/working/quick.csv

## 6. Final run, all 654 test images
Rediagnoses roles, hashes the data, resets the seed before each method, and saves versions, hardware, timings and raw latency samples.

In [ ]:
!python $R --mat $MAT --splits $SPLITS --revision $REV --methods fp32 minmax percentile ora_q --num-images 654 --seed 0 --rediagnose --hash-data --seed-per-method --role-report $E/role_report_final.json --out $E/results_final.csv

In [ ]:
import pandas as pd, json
df = pd.read_csv(f"{E}/results_final.csv")
print(df[["method", "quality_value", "abs_rel", "latency_median_ms", "peak_memory_mb"]])
print(json.dumps(json.load(open(f"{E}/role_report_final.json"))["operators"], indent=1))
meta = json.load(open(f"{E}/results_final_meta.json"))[-1]
print(json.dumps(meta["environment"], indent=1))
print({m: v["per_image_median_ms"] for m, v in meta["methods"].items()})

## 7. Optional: fc2 percentile override on the T4
Same as the CPU ablation, so it can be compared with the T4 rows. Run it only if step 6 finished.

In [ ]:
# !python $R --mat $MAT --splits $SPLITS --revision $REV --methods ora_q --num-images 654 --seed 0 --hash-data --seed-per-method --role-report $E/role_report_fc2_override.json --out $E/results_final_fc2_percentile.csv

## 8. Save
Download everything below, put it in `Quantization_and_Pruning/experiments/ora_q/` in your local clone, and stop the session. Do not copy the .mat files.

In [ ]:
!pip freeze > $E/pip_freeze_final.txt
!nvidia-smi > $E/nvidia_smi_final.txt
!cd $E && zip -q /kaggle/working/ora_q_final.zip results_final*.csv results_final*_meta.json role_report_final.json pip_freeze_final.txt nvidia_smi_final.txt && unzip -l /kaggle/working/ora_q_final.zip